In [13]:
import pandas as pd
from collections import defaultdict

df = pd.read_parquet("./dataset/train-00000-of-00003.parquet")
print(len(df))
#print(df.columns)
#print(df["text"].iloc[0])


215966


In [16]:
import regex as re

class BPEUtils:

    @staticmethod
    def pretokenize(text):
        pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        return re.findall(pattern, text)

    @staticmethod
    def count_pairs(ids, count_map):
        for pair in zip(ids, ids[1:]):
            count = count_map.get(pair, 0)
            count_map[pair] = count + 1

    @staticmethod
    def merge(ids, pair, new_id):
        merged_ids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
                merged_ids.append(new_id)
                i += 2
            else:
                merged_ids.append(ids[i])
                i += 1

        return merged_ids


In [34]:
ids_count = defaultdict(int)
for text in df["text"][:100]:
    for token in BPEUtils.pretokenize(text):        
        ids_count[tuple(token.encode('utf-8'))] += 1

top_10 = sorted(ids_count.items(), key=lambda x: x[1], reverse=True)[:10]
for key, value in top_10:
    print(f"meaning: {bytes(key).decode('utf-8')}, key: {key}, value: {value}")


meaning: 
, key: (10,), value: 8539
meaning: ., key: (46,), value: 7026
meaning: ,, key: (44,), value: 4473
meaning: ), key: (41,), value: 1868
meaning: (, key: (40,), value: 1640
meaning: 년, key: (235, 133, 132), value: 1569
meaning: 월, key: (236, 155, 148), value: 935
meaning:  (, key: (32, 40), value: 856
meaning:  -, key: (32, 45), value: 817
meaning: 일, key: (236, 157, 188), value: 733


In [40]:
import pickle

with open("output/bpe_rules.pkl", "rb") as f:
    rules = pickle.load(f)

vocab = {i: bytes([i]) for i in range(256)}
for (id1, id2), new_id in rules.items():
    vocab[new_id] = vocab[id1] + vocab[id2]

for key, value in list(rules.items())[1000:1050]:
    print(f"meaning: {vocab[value].decode('utf-8', errors='replace')}, key: {key}, value: {value}")


meaning:  러, key: (1142, 172), value: 1256
meaning: 10, key: (49, 48), value: 1257
meaning:  시대, key: (432, 347), value: 1258
meaning:  네, key: (785, 164), value: 1259
meaning: �, key: (236, 189), value: 1260
meaning: 헌, key: (699, 140), value: 1261
meaning:  주장, key: (464, 403), value: 1262
meaning: 존, key: (524, 180), value: 1263
meaning: 번째, key: (839, 913), value: 1264
meaning: 봉, key: (235, 1040), value: 1265
meaning:  이루, key: (304, 658), value: 1266
meaning:  존재, key: (1032, 562), value: 1267
meaning: 성을, key: (394, 282), value: 1268
meaning:  드, key: (440, 156), value: 1269
meaning:  

, key: (419, 10), value: 1270
meaning: 우스, key: (421, 323), value: 1271
meaning:  모든, key: (561, 966), value: 1272
meaning: 리를, key: (310, 314), value: 1273
meaning: ol, key: (111, 108), value: 1274
meaning:  �, key: (258, 168), value: 1275
meaning:  경제, key: (496, 405), value: 1276
meaning:  수도, key: (395, 319), value: 1277
meaning: 런, key: (396, 176), value: 1278
meaning: 리카, key: (310, 611), v